# Кольцевые коммуникации

Реализуйте три операции на CPU с Gloo. 
- `rank` — номер процесса,
- `world_size` — число процессов.
- Входные тензоры — одномерные CPU-тензоры одинакового размера на всех процессах. Их менять нельзя

In [1]:
%%writefile collectives.py
"""Кольцевые SUM-операции для одномерных CPU-тензоров и Gloo."""
import torch
import torch.distributed as dist


def ring_reduce_scatter(tensor: torch.Tensor) -> torch.Tensor:
    rank, world_size = dist.get_rank(), dist.get_world_size()
    if tensor.ndim != 1 or tensor.numel() % world_size:
        raise ValueError("Ожидается одномерный тензор длины, кратной world_size")
    chunks = [chunk.clone() for chunk in tensor.chunk(world_size)]
    left, right = (rank - 1) % world_size, (rank + 1) % world_size
    for step in range(world_size - 1):
        # Такой сдвиг оставляет суммарный блок с индексом rank у процесса rank.
        send_index = (rank - step - 1) % world_size
        recv_index = (rank - step - 2) % world_size
        received = torch.empty_like(chunks[recv_index])
        requests = dist.batch_isend_irecv([
            dist.P2POp(dist.isend, chunks[send_index], right),
            dist.P2POp(dist.irecv, received, left),
        ])
        for request in requests:
            request.wait()
        chunks[recv_index].add_(received)
    return chunks[rank]


def ring_all_gather(shard: torch.Tensor) -> torch.Tensor:
    rank, world_size = dist.get_rank(), dist.get_world_size()
    chunks = [torch.empty_like(shard) for _ in range(world_size)]
    chunks[rank].copy_(shard)
    left, right = (rank - 1) % world_size, (rank + 1) % world_size
    for step in range(world_size - 1):
        send_index = (rank - step) % world_size
        recv_index = (rank - step - 1) % world_size
        requests = dist.batch_isend_irecv([
            dist.P2POp(dist.isend, chunks[send_index], right),
            dist.P2POp(dist.irecv, chunks[recv_index], left),
        ])
        for request in requests:
            request.wait()
    return torch.cat(chunks)


def ring_all_reduce(tensor: torch.Tensor) -> torch.Tensor:
    return ring_all_gather(ring_reduce_scatter(tensor))


Overwriting collectives.py


## Сравнение с PyTorch

Сравним нашу реализацию с готовыми функциями торча `dist.reduce_scatter_tensor`, `dist.all_gather`
и `dist.all_reduce` на нескольких размерах тензоров. 

In [2]:
%%writefile check.py
from datetime import timedelta

import torch
import torch.distributed as dist

from collectives import ring_all_gather, ring_all_reduce, ring_reduce_scatter


torch.set_num_threads(1)
dist.init_process_group("gloo", timeout=timedelta(seconds=30))
try:
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    generator = torch.Generator().manual_seed(42 + rank)

    for chunk_size in (1, 7, 31):
        values = torch.randn(world_size * chunk_size, generator=generator, dtype=torch.float64)
        original = values.clone()
        expected_shard = torch.empty(chunk_size, dtype=values.dtype)
        dist.reduce_scatter_tensor(expected_shard, values.clone(), op=dist.ReduceOp.SUM)
        torch.testing.assert_close(ring_reduce_scatter(values), expected_shard)
        torch.testing.assert_close(values, original, rtol=0, atol=0)

        shard = torch.randn(chunk_size, generator=generator, dtype=torch.float64)
        original_shard = shard.clone()
        expected_parts = [torch.empty_like(shard) for _ in range(world_size)]
        dist.all_gather(expected_parts, shard)
        torch.testing.assert_close(ring_all_gather(shard), torch.cat(expected_parts))
        torch.testing.assert_close(shard, original_shard, rtol=0, atol=0)

        expected_sum = values.clone()
        dist.all_reduce(expected_sum, op=dist.ReduceOp.SUM)
        torch.testing.assert_close(ring_all_reduce(values), expected_sum)
        torch.testing.assert_close(values, original, rtol=0, atol=0)

    if rank == 0:
        print(f"OK: все три операции прошли проверки на {world_size} процессах.")
finally:
    dist.destroy_process_group()

Writing check.py


In [3]:
import sys

!{sys.executable} -m torch.distributed.run --standalone --nproc-per-node=2 check.py

OK: все три операции прошли проверки на 2 процессах.


In [4]:
!{sys.executable} -m torch.distributed.run --standalone --nproc-per-node=4 check.py

OK: все три операции прошли проверки на 4 процессах.
